# UNDERTONE - gemini_3_1_pro (closed model, Gemini API)

The frontier point the open roster cannot supply. Identical protocol - same
items, same rendered prompt, same letter-to-role randomisation, same ladder
windows - through the Gemini API, with one documented difference: the API
exposes no letter logits, so every cell is **scored by generation with strict
single-letter parsing** (`scorer: freegen`), thinking disabled so the rows
compare to the instruct models rather than the thinking variants.

CPU kernel: nothing here needs a GPU. Runs the ladder (L1-L4) on every item
pack attached - v1 (70), v2 (338), the 600 s band (85) - and resumes from
whatever `results/` already holds. Uploads are cached per audio, so the ~110
shared L3/L4 windows of a pack upload once.

The API key comes from Kaggle Secrets (`GEMINI_API_KEY`) or, failing that,
from a cell injected at push time; it is never in the repository.


In [ ]:
# Pinned for this model. If `load()` fails, this cell is the first thing to change.
%pip install -q "google-genai>=1.0.0"
%pip install -q "librosa>=0.10.2"
%pip install -q "soundfile>=0.12.1"
print("--- resolved versions (freeze these before the paper run) ---")
import importlib.metadata as md
for pkg in ["transformers", "accelerate", "torch", "librosa"]:
    try:
        print(f"{pkg:14s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:14s} not installed")

In [ ]:
import os, glob, json, random, sys
import numpy as np
SEED = 20260904
random.seed(SEED); np.random.seed(SEED)
# Kaggle Secrets first; a push-time injected cell may already have set it.
if not os.environ.get("GEMINI_API_KEY"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")
    except Exception as exc:
        print("no GEMINI_API_KEY in Kaggle Secrets:", exc)
assert os.environ.get("GEMINI_API_KEY"), "GEMINI_API_KEY is not set"
os.environ.setdefault("GEMINI_RPM", "20")
print("key set; RPM", os.environ["GEMINI_RPM"])

In [ ]:
REPO_URL = "https://github.com/DeepanIsCool/longaudiobench.git"
REPO_REF = "paper-run-21"   # pin to a commit sha before the paper run

import subprocess, shutil, os, sys
if os.path.exists("/kaggle/working/longaudiobench"):
    shutil.rmtree("/kaggle/working/longaudiobench")
for attempt in range(3):
    rc = subprocess.call(["git", "clone", "--depth", "1", "--branch", REPO_REF,
                          REPO_URL, "/kaggle/working/longaudiobench"])
    if rc == 0:
        break
else:
    raise RuntimeError("could not clone the benchmark repo")

# Which commit actually ran. REPO_REF may be a branch, and a branch moves: the
# results already on disk were produced by an unknown spread of commits, one of
# which switched the attention kernel, and nothing recorded it. Every row a
# sweep writes now carries this.
CODE_SHA = subprocess.run(["git", "-C", "/kaggle/working/longaudiobench",
                           "rev-parse", "HEAD"],
                          capture_output=True, text=True).stdout.strip()
os.environ["UNDERTONE_CODE_SHA"] = CODE_SHA
print(f"code: {REPO_REF} @ {CODE_SHA[:12]}")

sys.path.insert(0, "/kaggle/working/longaudiobench")
import importlib; importlib.invalidate_caches()

from undertone import ItemPack, adapters, env, runner, scoring
print("adapters registered:", len(adapters.list_adapters()))

if env.export_hf_token():
    print("HF token resolved")
elif globals().get("GATED"):
    raise RuntimeError(
        "this model is gated and no token was found. Add a Kaggle secret named "
        "HF_TOKEN, or write the token to .hf_token at the repo root.")

# O(n) attention instead of O(n^2). The math kernel materialises the full
# attention matrix; over ~45k audio tokens that is a 60 GiB allocation on a
# 15.6 GB card, and sharding across two T4s does not help because the matrix
# lives on one device. FlashAttention needs sm80+; this one runs on sm75.
print("attention backend:", env.prefer_memory_efficient_attention())

hw = env.resolve_hardware()
print(f"hardware: {hw.detail}  dtype={hw.dtype}  signature={hw.signature}")
print(f"versions: {env.versions()}")
# Every result row is stamped with this signature. The analysis refuses to put
# two signatures in one table -- a benchmark whose rows came from different
# backends compares machines, not models.

In [ ]:
sys.path.insert(0, "/kaggle/working/longaudiobench")
from undertone import ItemPack, adapters, runner

KEY = "gemini_3_1_pro"
OUT_ROOT = f"/kaggle/working/results/{KEY}"
os.makedirs(OUT_ROOT, exist_ok=True)
os.environ["GEMINI_UPLOAD_CACHE"] = f"{OUT_ROOT}/uploads.json"

packs = sorted(glob.glob("/kaggle/input/**/item_pack.jsonl", recursive=True))
assert packs, "attach the item-pack datasets (v1, v2, 600)"
adapter = adapters.get_adapter(KEY)
adapter.load()
print(KEY, "->", adapter.model_id, "| packs:", len(packs))

for path in packs:
    pack = ItemPack.load(path)
    pack_dir = os.path.dirname(path)
    out = f"{OUT_ROOT}/{pack.fingerprint}/results.jsonl"
    os.makedirs(os.path.dirname(out), exist_ok=True)
    print(f"\n=== pack {pack.fingerprint} ({len(pack)} items) -> {out}")
    runner.run_model(adapter, pack, out, conditions=["L1", "L2", "L3", "L4"],
                     audio_root=pack_dir, progress=True)
    rows = runner.load_rows(out)
    ok = [r for r in rows if not r.get("error") and r.get("role_chosen")]
    print(f"rows={len(rows)} valid={len(ok)} errors={len(rows)-len(ok)} api_calls={adapter.calls}")
    for c in ("L1", "L2", "L3", "L4"):
        s = [r for r in ok if r["condition"] == c]
        if s:
            print(f"  {c} n={len(s)} acc={sum(r['correct'] for r in s)/len(s):.3f} "
                  f"sal={sum(r['role_chosen']=='salience' for r in s)/len(s):.3f} "
                  f"abs={sum(r['role_chosen']=='absent' for r in s)/len(s):.3f}")
print("\nALL PACKS DONE")